# Custom Tools for Multi-Modal Processing

Extend agents with custom tools using the `@tool` decorator. This notebook demonstrates how to process multi-modal content including images, videos, and documents with custom tool implementations.

## What You'll Learn

- Create custom tools with the `@tool` decorator
- Process images, videos, and documents
- Handle multi-modal content in agent workflows
- Implement error handling and validation in tools

## Prerequisites

- Completed [Notebook 01: Hello World](01-hello-world-strands-agents.ipynb)
- Understanding of Python functions and decorators
- Sample media files (provided in `data-sample/` directory)

In [1]:
import boto3
from strands import Agent
from strands.models import BedrockModel
from strands.tools import tool
from datetime import datetime

print("✅ Imports successful!")

✅ Imports successful!


## Creating a Simple Tool

Let's create a simple calculator tool (using @tool decorator):

In [2]:
@tool
def calculator(operation: str, a: float, b: float) -> float:
    """Performs basic mathematical operations.
    
    Args:
        operation: The operation to perform (add, subtract, multiply, divide)
        a: First number
        b: Second number
    
    Returns:
        The result of the operation
    """
    operations = {
        "add": a + b,
        "subtract": a - b,
        "multiply": a * b,
        "divide": a / b if b != 0 else "Error: Division by zero"
    }
    return operations.get(operation, "Invalid operation")

print("✅ Calculator tool created!")

✅ Calculator tool created!


## Using Tools with Agents

Let's first create the model instance to be used by the agent

In [3]:
# Setup Bedrock model
session = boto3.Session(region_name='us-east-1')
bedrock_model = BedrockModel(
    model_id="us.amazon.nova-2-lite-v1:0",
    boto_session=session
)

Now let's create an agent that can use our calculator:

In [4]:
# Create agent with calculator tool
math_agent = Agent(
    model=bedrock_model,
    tools=[calculator],
    system_prompt="You are a helpful math assistant. Use the calculator tool to perform calculations."
)

print("✅ Math agent created with calculator tool!")

✅ Math agent created with calculator tool!


In [5]:
# Test the calculator tool
response = math_agent("What is 156 multiplied by 23?")
print(response)


Tool #1: calculator
156 multiplied by 23 equals **3,588**.156 multiplied by 23 equals **3,588**.



Now let's inspect the type AgentResult in Strands:

In [6]:
# Inspect the AgentResult object
print(f"Message: {response.message}")
print("-" * 100 + "\n")

print(f"Metrics: {response.metrics}")
print("-" * 100 + "\n")

print(f"State: {response.state}")
print("-" * 100 + "\n")

print(f"Stop Reason: {response.stop_reason}")
print("-" * 100 + "\n")

Message: {'role': 'assistant', 'content': [{'text': '156 multiplied by 23 equals **3,588**.'}]}
----------------------------------------------------------------------------------------------------

Metrics: EventLoopMetrics(cycle_count=2, tool_metrics={'calculator': ToolMetrics(tool={'toolUseId': 'tooluse_nSZXANWQ822rWnkcmUwrTO', 'name': 'calculator', 'input': {'operation': 'multiply', 'a': 156.0, 'b': 23.0}}, call_count=1, success_count=1, error_count=0, total_time=0.0)}, cycle_durations=[1.0145933628082275], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='b1fbde77-1ecc-4020-9d14-a3667c890fe7', usage={'inputTokens': 997, 'outputTokens': 45, 'totalTokens': 1042}), EventLoopCycleMetric(event_loop_cycle_id='51665704-88b1-4791-8e68-6b0ba1b5e052', usage={'inputTokens': 1071, 'outputTokens': 14, 'totalTokens': 1085})], usage={'inputTokens': 2068, 'outputTokens': 59, 'totalTokens': 2127})], traces=[<strands.telemetry.metrics.Trace object at 0x0000023632E6

## Creating More Complex Tools

Let's create a tool that gets current time information:

In [7]:
@tool
def get_current_time(timezone: str = "UTC") -> str:
    """Gets the current date and time.
    
    Args:
        timezone: The timezone (currently only UTC supported)
    
    Returns:
        Current date and time as a string
    """
    now = datetime.now()
    return f"Current time ({timezone}): {now.strftime('%Y-%m-%d %H:%M:%S')}"

print("✅ Time tool created!")

✅ Time tool created!


In [8]:
# Create agent with multiple tools
assistant = Agent(
    model=bedrock_model,
    tools=[calculator, get_current_time],
    system_prompt="You are a helpful assistant with access to calculator and time tools."
)

response = assistant("What time is it? Also, what's 50 plus 75?")
print(response)


Tool #1: get_current_time

Tool #2: calculator
The current time (UTC) is **2026-03-13 01:43:30**.

50 plus 75 equals **125**.The current time (UTC) is **2026-03-13 01:43:30**.

50 plus 75 equals **125**.



## Using Built-in Tools

Strands Agents includes pre-built tools for common tasks:

In [9]:
from strands_tools import image_reader, file_read 
# Example of video_reader tool structure
# (This is already implemented in video_reader.py)

from video_reader_local import video_reader_local

# Create agent with built-in tools
multimodal_agent = Agent(
    model=bedrock_model,
    tools=[image_reader, file_read,video_reader_local],
    system_prompt="""You are a multi-modal assistant that can:
    - Read and analyze images
    - Process documents (PDF, CSV, DOCX, etc.)
    - Use advanced reasoning for complex tasks.
    - Analyze videos and provide detailed insights.
    """
)

print("✅ Multi-modal agent created with built-in tools!")

✅ Multi-modal agent created with built-in tools!


We can see which tools are loaded in our agent in `agent.tool_name`, along with a JSON representation of the tools in `agent.tool_config` that also includes the tool descriptions and input parameters

In [10]:
print(multimodal_agent.tool_names)

print(multimodal_agent.tool_registry.get_all_tools_config())

['image_reader', 'file_read', 'video_reader_local']
{'image_reader': {'name': 'image_reader', 'description': 'Reads an image file from a given path and returns it in the format required for the Converse API', 'inputSchema': {'json': {'type': 'object', 'properties': {'image_path': {'type': 'string', 'description': 'The path to the image file'}}, 'required': ['image_path']}}}, 'file_read': {'name': 'file_read', 'description': 'File reading tool with search capabilities, various reading modes, and document mode support for Bedrock compatibility.\n\nFeatures:\n1. Multi-file support (comma-separated paths)\n2. Full document format support (pdf, doc, docx, etc.)\n3. Search and filtering capabilities\n4. Version control integration\n5. Document block generation for Bedrock\n\nModes:\n- find: List matching files\n- view: Display file contents\n- lines: Show specific line ranges\n- chunk: Read byte chunks\n- search: Pattern searching\n- stats: File statistics\n- preview: Quick content preview\n

In [11]:
# Example 1: Image analysis
print("=== 📸 IMAGE ANALYSIS ===")
image_result = multimodal_agent("Analyze the image data-sample/diagram.jpg in detail and describe everything you observe")
# print(image_result)
print("\n" + "="*80 + "\n")

=== 📸 IMAGE ANALYSIS ===

Tool #1: image_reader
# Analysis of the Diagram: "Private Assistant V2" Architecture

## Overview
This diagram illustrates the architecture of a "Private Assistant V2" system, showing how it processes user messages through various AWS services to generate responses. The system appears to be designed for handling multi-modal inputs (text, audio, images, documents, video) from users via WhatsApp messaging.

## Key Components and Flow

### **1. User Interaction Layer**
- **User**: The starting point, represented by a WhatsApp icon, indicating users interact through WhatsApp messaging.
- **AWS End User Messaging Social**: Acts as the interface between users and the system, handling incoming messages from WhatsApp.

### **2. Message Routing**
- **SNS Topic Inbound Message**: Receives messages from the AWS End User Messaging service and routes them based on message type.
- The SNS topic triggers different processing paths depending on the message type:
  - **Text**:

In [12]:
# Example 2: Document analysis (if you have a PDF document)
print("=== 📄 DOCUMENT ANALYSIS ===")
doc_result = multimodal_agent("Summarize as json the content of the document data-sample/Welcome-Strands-Agents-SDK.pdf")
# print(doc_result)

=== 📄 DOCUMENT ANALYSIS ===

Tool #2: file_read


╭───────────────────────────────────────────────────── Error ─────────────────────────────────────────────────────╮
│ Error reading file data-sample/Welcome-Strands-Agents-SDK.pdf: 'charmap' codec can't decode byte 0x81 in        │
│ position 1695: character maps to <undefined>                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

I apologize for the error. It seems there was an issue reading the PDF file `data-sample/Welcome-Strands-Agents-SDK.pdf`. The error indicates an encoding problem with the file, specifically a "charmap" codec issue where it couldn't decode a particular byte (0x81 at position 1695).

This type of error typically occurs when:
1. The PDF file has encoding issues or corruption
2. The file is not properly encoded in a format that the system can process
3. There might be special characters or binary data that the text encoding can't handle

Unfortunately, I'm unable to directly summarize the content of this PDF due to this encoding error. To get a summary, you would need to:

1. Try opening the PDF in a PDF reader to see if it displays correctly
2. Re-save or convert the PDF to ensure it's properly encoded (UTF-8 is often a good choice)
3. If the file is corrupted, try obtaining a fresh copy

Would you like me to help with anything else regarding this file, or perhaps you could share the cont

In [13]:
# Example 2: Video analysis
print("=== 🎬 VIDEO ANALYSIS ===")
video_result = multimodal_agent("Analyze the video data-sample/climbing-video.mp4 and describe in detail the actions and scenes you observe")
print(video_result)
print("\n" + "="*80 + "\n")

=== 🎬 VIDEO ANALYSIS ===

Tool #3: video_reader_local
# Video Analysis: Climbing Video

## Detailed Observation

Based on the analysis of the climbing video (`data-sample/climbing-video.mp4`), here are the key observations:

### **Primary Action**
The video shows a person engaged in **mountain climbing**, specifically **ascending an icy mountain**. The climber is using specialized climbing equipment to navigate the challenging terrain.

### **Key Elements Observed**

1. **Climber's Equipment and Technique**
   - The climber is wearing a **harness** with attached **ropes** for safety
   - The ropes are properly secured, indicating adherence to climbing safety protocols
   - The climber is carefully **placing feet on the ice surface**, showing deliberate and cautious movement

2. **Environment**
   - The setting is clearly an **icy mountain**, suggesting this is a **glacier climbing** or **ice climbing** scenario
   - The ice surface appears to be the primary terrain being navigated

3. 

In [14]:
# Inspect the AgentResult object
print(f"Message: {video_result.message}")
print("-" * 100 + "\n")

print(f"Metrics: {video_result.metrics}")
print("-" * 100 + "\n")

print(f"State: {video_result.state}")
print("-" * 100 + "\n")

print(f"Stop Reason: {video_result.stop_reason}")
print("-" * 100 + "\n")

Message: {'role': 'assistant', 'content': [{'text': "# Video Analysis: Climbing Video\n\n## Detailed Observation\n\nBased on the analysis of the climbing video (`data-sample/climbing-video.mp4`), here are the key observations:\n\n### **Primary Action**\nThe video shows a person engaged in **mountain climbing**, specifically **ascending an icy mountain**. The climber is using specialized climbing equipment to navigate the challenging terrain.\n\n### **Key Elements Observed**\n\n1. **Climber's Equipment and Technique**\n   - The climber is wearing a **harness** with attached **ropes** for safety\n   - The ropes are properly secured, indicating adherence to climbing safety protocols\n   - The climber is carefully **placing feet on the ice surface**, showing deliberate and cautious movement\n\n2. **Environment**\n   - The setting is clearly an **icy mountain**, suggesting this is a **glacier climbing** or **ice climbing** scenario\n   - The ice surface appears to be the primary terrain bei

## Direct Tool Usage

You can also call tools directly from the agent:

In [15]:
print(multimodal_agent.tool_names)

['image_reader', 'file_read', 'video_reader_local']


In [16]:
# Example 4. Direct use of tools
video_analysis = multimodal_agent.tool.video_reader_local(
     video_path="data-sample/climbing-video.mp4", 
     text_prompt="What are the main elements in this video?"
)

In [17]:
print(video_analysis)

{'status': 'success', 'content': [{'text': '🎥 Video Analysis Results:\n\n**Analysis:** The person is climbing up the mountain with a rope in their hand. They are holding onto the rope for support and stability as they ascend the steep and rocky terrain.\n\n---\n**Technical Details:**\n- Model Used: us.amazon.nova-pro-v1:0\n- Region: us-west-2\n- Video Path: data-sample/climbing-video.mp4\n- File Size: 5.01MB\n- Processing: Local (no S3 upload)\n'}], 'toolUseId': 'tooluse_video_reader_local_489014181'}


In [18]:
print(video_analysis['status'])
print("-" * 100 + "\n")
print(video_analysis['content'][0]['text'])
print("-" * 100 + "\n")
print(video_analysis['toolUseId'])
print("-" * 100 + "\n")

success
----------------------------------------------------------------------------------------------------

🎥 Video Analysis Results:

**Analysis:** The person is climbing up the mountain with a rope in their hand. They are holding onto the rope for support and stability as they ascend the steep and rocky terrain.

---
**Technical Details:**
- Model Used: us.amazon.nova-pro-v1:0
- Region: us-west-2
- Video Path: data-sample/climbing-video.mp4
- File Size: 5.01MB
- Processing: Local (no S3 upload)

----------------------------------------------------------------------------------------------------

tooluse_video_reader_local_489014181
----------------------------------------------------------------------------------------------------



### Additional Samples
An agent that uses the video reader using a AWS S3 bucket for larger videos. 

For that you need to add the bucket environment variable

```bash
export VIDEO_READER_S3_BUCKET = "YOU-BUCKET-NAME"
```

In [19]:
from strands_tools import image_reader, file_read 
# Example of video_reader tool structure
# (This is already implemented in video_reader.py)

from video_reader import video_reader


# Create agent with built-in tools
multimodal_agent = Agent(
    model=bedrock_model,
    tools=[image_reader, file_read,video_reader],
    system_prompt="""You are a multi-modal assistant that can:
    - Read and analyze images
    - Process documents (PDF, CSV, DOCX, etc.)
    - Use advanced reasoning for complex tasks.
    - Analyze videos and provide detailed insights.
    """
)

print("✅ Multi-modal agent created with built-in tools!")

✅ Multi-modal agent created with built-in tools!


In [ ]:
# Example 4. Direct use of tools
from video_reader_local import video_reader_local

video_analysis = video_reader_local(
     video_path="check in my aws s3 bucket", 
     text_prompt="What are the main elements in this video?"
)

In [31]:
print(video_analysis)

{'status': 'error', 'content': [{'text': '❌ Video file not found: check in my aws s3 bucket demo-bucket-827739414024-us-east-1-an'}]}


## Summary

In this notebook, you learned:

✅ How to create custom tools with the `@tool` decorator

✅ How to add tools to agents

✅ How to use built-in tools from strands_tools

✅ How to create agents with multiple tools

✅ How to call tools directly


### Next Steps

Continue to the next notebook to learn about Model Context Protocol (MCP) integration!